In [1]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand(size=(2, 4))
net(X)

tensor([[0.1327],
        [0.1110]], grad_fn=<AddmmBackward0>)

参数访问

通过Sequential类定义模型时， 我们可以通过索引来访问模型的任意层。下面输出结果为该层的权重和偏置

In [2]:
print(net[2].state_dict())

OrderedDict([('weight', tensor([[-0.1326,  0.1889,  0.0386,  0.1909, -0.2479, -0.1039, -0.2912,  0.2225]])), ('bias', tensor([0.1019]))])


目标参数

每个参数都表示为参数类的一个实例，下面的代码从第二个全连接层（即第三个神经网络层）提取偏置， 提取后返回的是一个参数类实例，并进一步访问该参数的值。

In [3]:
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([0.1019], requires_grad=True)
tensor([0.1019])


还可以访问每个参数的梯度，但是上面的网络没有进行反向传播，梯度处于初始状态

In [4]:
net[2].weight.grad == None

True

一次性访问所有参数

下面代码比较访问第一个全连接层的参数和访问所有层。

In [5]:
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


从嵌套块收集参数

将多个块相互嵌套，参数命名约定是如何工作的

In [6]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        # 在这里嵌套
        net.add_module(f'block {i}', block1())
    return net

rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)

tensor([[-0.1309],
        [-0.1310]], grad_fn=<AddmmBackward0>)

In [7]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


可以像通过嵌套列表索引一样访问每个层的参数

In [8]:
rgnet[0][1][0].bias.data

tensor([-0.4625, -0.2931, -0.4961, -0.1803, -0.4076,  0.4220,  0.0865,  0.2830])

参数初始化

深度学习框架提供默认随机初始化， 也允许我们创建自定义初始化方法， 满足我们通过其他规则实现初始化权重。

1.内置初始化

下面的代码将所有权重参数初始化为标准差为0.01的高斯随机变量， 且将偏置参数设置为0。

In [9]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([ 0.0093,  0.0262, -0.0147,  0.0127]), tensor(0.))

将所有参数初始化为给定的常数，比如初始化为1。

In [10]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)
net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

2.对某些块应用不同的初始化方法

下面我们使用Xavier初始化方法初始化第一个神经网络层， 然后将第三个神经网络层初始化为常量值42。

In [11]:
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)

net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([-0.3623,  0.1887, -0.4266,  0.5247])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


3.自定义初始化

在下面的例子中，我们使用以下的分布为任意权重参数$w$定义初始化方法：

$$
\begin{aligned}
    w \sim \begin{cases}
        U(5, 10) & \text{ 可能性 } \frac{1}{4} \\
            0    & \text{ 可能性 } \frac{1}{2} \\
        U(-10, -5) & \text{ 可能性 } \frac{1}{4}
    \end{cases}
\end{aligned}
$$


In [12]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape)
                        for name, param in m.named_parameters()][0])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[ 9.5204, -0.0000, -0.0000,  0.0000],
        [ 9.9429,  0.0000, -7.9759,  0.0000]], grad_fn=<SliceBackward0>)

In [13]:
net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42
net[0].weight.data[0]

tensor([42.,  1.,  1.,  1.])

参数绑定

在多个层共享参数，可以定义一个稠密层，然后使用它的参数来设置另一个层的参数。

In [14]:
# 我们需要给共享层一个名称，以便可以引用它的参数
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.Linear(8, 1))
net(X)
# 检查参数是否相同
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
# 确保它们实际上是同一个对象，而不只是有相同的值
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


## 练习

1. 使用 :numref:`sec_model_construction` 中定义的`FancyMLP`模型，访问各个层的参数。

In [15]:
import torch
from torch import nn
import torch.nn.functional as F

class FancyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        X = self.linear(X)   # 重复使用同一个全连接层
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

查看整个模型的参数

In [16]:
net = FancyMLP()
print(net.parameters())

<generator object Module.parameters at 0x000002B165187580>


访问全连接层的权重和偏置

In [17]:
print(net.linear.weight)
print(net.linear.bias)

Parameter containing:
tensor([[-4.6707e-02,  2.2163e-01, -1.2918e-01,  1.4710e-01,  1.9894e-01,
          1.8832e-01, -7.6787e-03,  1.6681e-02, -1.7116e-01, -8.5410e-02,
         -1.5015e-01,  2.7076e-02,  1.1473e-01, -1.7069e-01, -2.9865e-02,
         -1.3214e-01,  4.5232e-02, -1.9039e-01,  1.5534e-02, -8.1851e-02],
        [ 4.7593e-02, -5.4285e-03, -5.1521e-02,  8.9035e-02, -1.5423e-01,
         -9.5988e-02,  1.9206e-01,  9.3104e-02,  1.4903e-01,  1.4579e-01,
         -2.1677e-01,  2.1205e-01,  1.5578e-01,  7.6909e-02,  2.1304e-01,
         -1.4845e-01, -1.0971e-01, -1.6680e-01, -1.5303e-01,  7.2396e-02],
        [ 2.0734e-01, -1.1210e-01,  4.2015e-02,  7.3797e-02, -1.8965e-01,
          9.0920e-02,  4.4105e-02, -9.3356e-05,  9.3593e-02,  2.0920e-01,
          1.8902e-01, -1.2219e-01, -7.5155e-02, -9.7437e-03, -1.4738e-01,
          1.5514e-01,  2.1529e-01,  1.8801e-01,  2.3630e-02, -1.8098e-01],
        [-1.1176e-01,  4.0127e-02,  1.1207e-01, -4.8881e-02,  6.6471e-02,
         -9.4

1. 查看初始化模块文档以了解不同的初始化方法。

In [ ]:
# 1. 正态分布初始化
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)

# 2. 均匀分布初始化
def init_uniform(m):
    if type(m) == nn.Linear:
        nn.init.uniform_(m.weight, -0.1, 0.1)
        nn.init.zeros_(m.bias)

# 3. 常数初始化
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 0.5)
        nn.init.zeros_(m.bias)

# 4. Xavier 初始化（适合 tanh/sigmoid）
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)

# 5. Kaiming 初始化（适合 ReLU）
def init_kaiming(m):
    if type(m) == nn.Linear:
        nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')

# 6. 正交初始化
def init_orthogonal(m):
    if type(m) == nn.Linear:
        nn.init.orthogonal_(m.weight)

"""
常见初始化方法总结：

1. normal_       → 正态分布，简单基础方法
2. uniform_      → 均匀分布
3. constant_     → 常数初始化（一般用于bias）
4. xavier_*      → 适合 sigmoid / tanh
5. kaiming_*     → 适合 ReLU（最常用）
6. orthogonal_   → 深层网络/保持稳定性


自定义初始化函数

In [ ]:
def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)

1. 构建包含共享参数层的多层感知机并对其进行训练。在训练过程中，观察模型各层的参数和梯度。

同一个层被调用两次，其中权重和偏置共享一份，一个参数被多次用到，它收到的梯度是所有使用路径共同贡献的总和，参数更新后所有使用该层的地方都会同时改变。

1. 为什么共享参数是个好主意？

共享参数可以减少模型参数量，降低内存和计算开销，同时减少过拟合风险。更重要的是，它能够利用任务中的重复结构，例如图像中不同位置的局部模式、序列中不同时间步的相似规律，从而提高模型的泛化能力。